<a href="https://colab.research.google.com/github/aman305/AI_tech_learning/blob/main/vector_store_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain_community

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash-lite', 'google/gemin…

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_classic.vectorstores import Chroma
from langchain_classic.schema import Document

In [ ]:
doc1 = Document(
     page_content="Python is a programming language.",
        metadata={"topic": "Python"}
)

doc2 = Document(
    page_content="Java is an object-oriented programming language.",
    metadata={"topic": "Java"}
)

doc3 = Document(
       page_content="LangChain simplifies AI application development.",
        metadata={"topic": "AI"}
)

In [17]:
import os

# IMPORTANT: Replace "YOUR_OPENAI_API_KEY" with your actual OpenAI API key.

vector_store = Chroma(
    embedding_function=OpenAIEmbeddings(),
    persist_directory="my_chromadb",
    collection_name="sample"
)

In [18]:
vector_store.add_documents([doc1, doc2, doc3])
print("Documents added to ChromaDB. The 'my_chromadb' directory should now be visible.")

Documents added to ChromaDB. The 'my_chromadb' directory should now be visible.


In [ ]:
vector_store.persist()
print("ChromaDB has persisted the data. You should now see the 'my_chromadb' directory.")

ChromaDB has persisted the data. You should now see the 'my_chromadb' directory.


/tmp/ipykernel_649/3596426328.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()


In [ ]:
import os

directory_path = "my_chromadb"
if os.path.exists(directory_path):
    print(f"The directory '{directory_path}' exists.")
    print(f"Contents of '{directory_path}': {os.listdir(directory_path)}")
else:
    print(f"The directory '{directory_path}' does NOT exist.")

The directory 'my_chromadb' exists.
Contents of 'my_chromadb': ['a28fcf0a-ca4b-4ce7-b687-e314ec594d57', 'chroma.sqlite3']


In [20]:
vector_store.get(include=["embeddings", "documents", "metadatas"])

{'ids': ['ff278715-75a0-48b7-9f3a-06aea30e31d0',
  '79e2f2f6-729a-4163-8668-caa318321e4a',
  'd843a261-8ddc-46d3-a873-8445aac086a1',
  '5951ff86-68bf-473f-97c7-f13295d9ba83',
  '0f3e8080-c4fd-4bc1-9fed-b4d7ee58daa2',
  'c7a5e1f1-1445-4684-adf7-444e385d6f51',
  'c6fc3d00-960e-4912-b14a-d569395c89e0',
  'f2fa2383-eac1-4432-8d04-7c5ef06c2abd',
  'a4144e91-fa00-4b29-8fae-3c6ce845bb45'],
 'embeddings': array([[ 0.0062781 , -0.0094983 ,  0.00577819, ...,  0.00373959,
          0.00366168, -0.03736995],
        [ 0.00489209,  0.00261007,  0.00166732, ..., -0.01105817,
          0.00041365, -0.03156929],
        [-0.00601396,  0.01284896, -0.01967695, ..., -0.01463841,
         -0.00863849, -0.02861719],
        ...,
        [ 0.0062781 , -0.0094983 ,  0.00577819, ...,  0.00373959,
          0.00366168, -0.03736995],
        [ 0.00489209,  0.00261007,  0.00166732, ..., -0.01105817,
          0.00041365, -0.03156929],
        [-0.00601396,  0.01284896, -0.01967695, ..., -0.01463841,
         -0

In [22]:
vector_store.similarity_search(query="What is langchain", k=2)

[Document(metadata={'topic': 'AI'}, page_content='LangChain simplifies AI application development.'),
 Document(metadata={'topic': 'AI'}, page_content='LangChain simplifies AI application development.')]

In [23]:
vector_store.similarity_search_with_score(query="what is langchain", k = 2)

[(Document(metadata={'topic': 'AI'}, page_content='LangChain simplifies AI application development.'),
  0.4377708435058594),
 (Document(metadata={'topic': 'AI'}, page_content='LangChain simplifies AI application development.'),
  0.4377708435058594)]

In [24]:
# meta-data filtering
vector_store.similarity_search(
    query="",
    filter={"topic":"AI"}
)

[Document(metadata={'topic': 'AI'}, page_content='LangChain simplifies AI application development.'),
 Document(metadata={'topic': 'AI'}, page_content='LangChain simplifies AI application development.'),
 Document(metadata={'topic': 'AI'}, page_content='LangChain simplifies AI application development.')]

In [26]:
update_doc_py = Document(
    page_content="Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, an extensive standard library, and garbage collection. ",
    metadata={"topic":"python"}
)

vector_store.update_document(
    document_id='ff278715-75a0-48b7-9f3a-06aea30e31d0',
    document=update_doc_py
)

In [27]:
vector_store.get(include=["embeddings", "documents", "metadatas"])

{'ids': ['ff278715-75a0-48b7-9f3a-06aea30e31d0',
  '79e2f2f6-729a-4163-8668-caa318321e4a',
  'd843a261-8ddc-46d3-a873-8445aac086a1',
  '5951ff86-68bf-473f-97c7-f13295d9ba83',
  '0f3e8080-c4fd-4bc1-9fed-b4d7ee58daa2',
  'c7a5e1f1-1445-4684-adf7-444e385d6f51',
  'c6fc3d00-960e-4912-b14a-d569395c89e0',
  'f2fa2383-eac1-4432-8d04-7c5ef06c2abd',
  'a4144e91-fa00-4b29-8fae-3c6ce845bb45'],
 'embeddings': array([[ 0.00285219, -0.00476949,  0.00674383, ..., -0.0124799 ,
         -0.00094201, -0.03508822],
        [ 0.00489209,  0.00261007,  0.00166732, ..., -0.01105817,
          0.00041365, -0.03156929],
        [-0.00601396,  0.01284896, -0.01967695, ..., -0.01463841,
         -0.00863849, -0.02861719],
        ...,
        [ 0.0062781 , -0.0094983 ,  0.00577819, ...,  0.00373959,
          0.00366168, -0.03736995],
        [ 0.00489209,  0.00261007,  0.00166732, ..., -0.01105817,
          0.00041365, -0.03156929],
        [-0.00601396,  0.01284896, -0.01967695, ..., -0.01463841,
         -0

In [29]:
vector_store.delete(ids=['c6fc3d00-960e-4912-b14a-d569395c89e0'])

In [30]:
vector_store.get(include=["embeddings", "documents", "metadatas"])

{'ids': ['ff278715-75a0-48b7-9f3a-06aea30e31d0',
  '79e2f2f6-729a-4163-8668-caa318321e4a',
  'd843a261-8ddc-46d3-a873-8445aac086a1',
  '5951ff86-68bf-473f-97c7-f13295d9ba83',
  '0f3e8080-c4fd-4bc1-9fed-b4d7ee58daa2',
  'c7a5e1f1-1445-4684-adf7-444e385d6f51',
  'f2fa2383-eac1-4432-8d04-7c5ef06c2abd',
  'a4144e91-fa00-4b29-8fae-3c6ce845bb45'],
 'embeddings': array([[ 0.00285219, -0.00476949,  0.00674383, ..., -0.0124799 ,
         -0.00094201, -0.03508822],
        [ 0.00489209,  0.00261007,  0.00166732, ..., -0.01105817,
          0.00041365, -0.03156929],
        [-0.00601396,  0.01284896, -0.01967695, ..., -0.01463841,
         -0.00863849, -0.02861719],
        ...,
        [-0.00601396,  0.01284896, -0.01967695, ..., -0.01463841,
         -0.00863849, -0.02861719],
        [ 0.00489209,  0.00261007,  0.00166732, ..., -0.01105817,
          0.00041365, -0.03156929],
        [-0.00601396,  0.01284896, -0.01967695, ..., -0.01463841,
         -0.00863849, -0.02861719]]),
 'documents': [

### Using FAISS Vector Store
FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. It's often used when you need to handle very large datasets of embeddings efficiently.

In [ ]:
!pip install faiss-cpu

First, import the necessary classes from LangChain and create a FAISS vector store. We'll use the same `OpenAIEmbeddings` for consistency, but you can use other embedding functions as well.

Unlike Chroma, FAISS doesn't inherently handle persistence in the same file-based manner. You typically save and load the entire FAISS index if you want to persist it.

In [ ]:
from langchain_community.vectorstores import FAISS

# Create a FAISS vector store
# Note: FAISS typically doesn't take a persist_directory directly during initialization like Chroma
faiss_vector_store = FAISS.from_documents(
    documents=[doc1, doc2, doc3], # Using the same documents as before
    embedding=OpenAIEmbeddings()
)

print("FAISS vector store created and documents added.")

### Performing a Similarity Search with FAISS
Similar to Chroma, you can perform a similarity search to find documents that are semantically close to your query.

In [ ]:
faiss_query = "What is Python used for?"
faiss_results = faiss_vector_store.similarity_search(faiss_query, k=2)

print(f"Query: {faiss_query}\n")
print("Top 2 similar documents from FAISS:")
for i, doc in enumerate(faiss_results):
    print(f"Document {i+1}:")
    print(f"  Content: {doc.page_content}")
    print(f"  Metadata: {doc.metadata}")


### Persisting and Loading FAISS Indexes
To save and load your FAISS index, you use its `save_local` and `load_local` methods. This creates a directory containing the index files.

In [ ]:
faiss_index_path = "my_faiss_index"
faiss_vector_store.save_local(faiss_index_path)
print(f"FAISS index saved to '{faiss_index_path}'")

# To load the index later
loaded_faiss_vector_store = FAISS.load_local(faiss_index_path, OpenAIEmbeddings())
print(f"FAISS index loaded from '{faiss_index_path}'")

# You can then perform searches on the loaded index
loaded_results = loaded_faiss_vector_store.similarity_search("programming language examples", k=1)
print("\nSearch on loaded FAISS index:")
for i, doc in enumerate(loaded_results):
    print(f"Document {i+1}:")
    print(f"  Content: {doc.page_content}")
    print(f"  Metadata: {doc.metadata}")


### Chroma vs. FAISS Vector Stores

Both Chroma and FAISS are used for efficient similarity searches on vector embeddings, but they have key differences:

*   **Persistence:**
    *   **Chroma:** Designed with built-in persistence. When you initialize Chroma with a `persist_directory`, it automatically saves your data (embeddings, documents, and metadata) to disk in a structured format (like a SQLite database). Loading is also straightforward from this directory.
    *   **FAISS:** Primarily an in-memory library for similarity search. While it's extremely fast, it does not inherently persist data on its own. To save a FAISS index, you need to explicitly use methods like `save_local()` and `load_local()`, which serialize the index to disk. The original documents and metadata are not saved by FAISS itself and would need to be managed separately if you want to retrieve them alongside the vector index.

*   **Metadata Handling:**
    *   **Chroma:** Excellent support for metadata. You can store rich metadata alongside your documents and use it for filtering during similarity searches, as demonstrated earlier.
    *   **FAISS:** Does not directly store metadata with the vectors. When you add documents, FAISS only uses the embeddings. If you need metadata for retrieved results, you typically need to manage a separate mapping (e.g., a dictionary or database) between the FAISS index's internal IDs and your original documents/metadata.

*   **Scalability & Deployment:**
    *   **Chroma:** Can be deployed in various modes, including client-server setups, which allows for more complex, distributed applications. It's often seen as a more complete, out-of-the-box solution for vector databases.
    *   **FAISS:** Very powerful for local, in-memory indexing, especially for very large datasets. It's a lower-level library focused purely on efficient similarity search algorithms. For distributed or managed deployments, you would typically integrate FAISS into a larger system.

*   **Ease of Use (Initial Setup):**
    *   **Chroma:** Often considered more user-friendly for getting started, especially with its automatic persistence and metadata handling.
    *   **FAISS:** Requires a bit more manual management, particularly around persistence and metadata, but offers more fine-grained control over the indexing process and algorithms.

In summary, if you need easy persistence, rich metadata filtering, and a more complete vector database experience, Chroma is a strong choice. If you prioritize raw speed for similarity search on large, static datasets and are comfortable managing persistence and metadata externally, FAISS is an excellent, high-performance option.